In [20]:
import numpy as np

class SevenWondersDecoder:
    def __init__(self, card_catalog, wonder_catalog, token_catalog):
        self.card_catalog = card_catalog
        self.wonder_catalog = wonder_catalog
        self.token_catalog = token_catalog
        self.num_cards = 73

    def decode(self, vector):
        res = []
        res.append("=== STAN GLOBALNY ===")
        res.append(f"Aktywny Gracz: {'Gracz 1' if vector[0] > 0.5 else 'Gracz 2'}")
        
        epoki = ["I", "II", "III"]
        epoka = next((epoki[i] for i in range(3) if vector[2+i] > 0.5), "Nieznana")
        res.append(f"Epoka: {epoka}")
        res.append(f"Pozycja Konfliktu: {vector[5] * 18 - 9:.0f}")
        res.append(f"Koniec Gry: {bool(vector[6])}")

        # Sekcje Graczy
        res.append(self._decode_player(vector, 48, "AKTYWNY GRACZ"))
        res.append(self._decode_player(vector, 169, "PRZECIWNIK"))

        # Piramida (od indeksu 290)
        res.append("\n=== PIRAMIDA ===")
        # slot_size = 4 + self.num_cards
        # start_idx = 290
        # for i in range(20):
        #     idx = start_idx + (i * slot_size)
        #     if vector[idx] > 0.5: # IsPresent
        #         status = "Dostępna" if vector[idx+2] > 0.5 else ("Zakryta" if vector[idx+1] > 0.5 else "Odkryta")
        #         koszt = vector[idx+3] * 20 # Zakładając MaxCoins = 20
                
        #         # Szukanie nazwy karty
        #         card_bits = vector[idx+4 : idx+4+self.num_cards]
        #         card_name = "Nieznana"
        #         if any(card_bits > 0.5):
        #             card_name = self.card_catalog[np.argmax(card_bits)]
                
        #         res.append(f"Slot {i:2d}: {card_name:20} | {status:10} | Koszt: {koszt:2.0f}")

        return "\n".join(res)

    def _decode_player(self, vector, start_idx, label):
        p = [f"\n--- {label} ---"]
        p.append(f"Monety: {vector[start_idx] * 100:.0f} | PZ: {vector[start_idx+1] * 100:.0f}")
        
        # Surowce (G, K, D, S, P)
        s = vector[start_idx+2 : start_idx+7]
        p.append(f"Surowce: Glina:{s[0]*10:.0f} Kamień:{s[1]*10:.0f} Drewno:{s[2]*10:.0f} Szkło:{s[3]*10:.0f} Papirus:{s[4]*10:.0f}")
        
        # Cuda (posiadane/zbudowane)
        w_start = start_idx + 13 + self.num_cards + 7 # Przesunięcie po surowcach, nauce i kartach
        # built_wonders = []
        # for i in range(len(self.wonder_catalog)):
        #     w_idx = w_start + (i * 2)
        #     if vector[w_idx] > 0.5: # Owned
        #         status = "[ZBUDOWANO]" if vector[w_idx+1] > 0.5 else "[W PULI]"
        #         built_wonders.append(f"{self.wonder_catalog[i]} {status}")
        # p.append("Cuda: " + (", ".join(built_wonders) if built_wonders else "Brak"))
        
        return "\n".join(p)

# PRZYKŁAD UŻYCIA
# Musisz podać listy nazw w tej samej kolejności co w C#
cards = ["Tartak", "Kamieniołom", "Glinianka", "..."] # Pełna lista 73 kart
wonders = ["Piramidy", "Wiszące Ogrody", "..."] # 12 cudów
tokens = ["Rolnictwo", "Filozofia", "Prawo", "Strategia", "Matematyka", 
          "Architektura", "Budownictwo", "Urbanistyka", "Ekonomia", "Teologia"] # 10 żetonów

decoder = SevenWondersDecoder(cards, wonders, tokens)

In [22]:
import glob
import os

list_of_files = glob.glob('../../GameTests/EncoderResults/*.txt') 
# list_of_files = glob.glob('C:/Users/kubeu/Kuba-dokumenty/Magisterka/7 Wonders/GameTests/EncoderResults/*.txt')
for file in list_of_files:    
    print(f"Znaleziono plik: {file}")
    raw_vector = np.loadtxt(file)
    print(decoder.decode(raw_vector))

Znaleziono plik: ../../GameTests/EncoderResults\vector_state.txt
=== STAN GLOBALNY ===
Aktywny Gracz: Gracz 1
Epoka: I
Pozycja Konfliktu: 0
Koniec Gry: False

--- AKTYWNY GRACZ ---
Monety: 7 | PZ: 0
Surowce: Glina:0 Kamień:0 Drewno:0 Szkło:0 Papirus:0

--- PRZECIWNIK ---
Monety: 7 | PZ: 0
Surowce: Glina:0 Kamień:0 Drewno:0 Szkło:0 Papirus:0

=== PIRAMIDA ===
Znaleziono plik: ../../GameTests/EncoderResults\vector_state_after_move.txt
=== STAN GLOBALNY ===
Aktywny Gracz: Gracz 2
Epoka: I
Pozycja Konfliktu: 1
Koniec Gry: False

--- AKTYWNY GRACZ ---
Monety: 7 | PZ: 0
Surowce: Glina:0 Kamień:0 Drewno:0 Szkło:0 Papirus:0

--- PRZECIWNIK ---
Monety: 7 | PZ: 0
Surowce: Glina:0 Kamień:0 Drewno:0 Szkło:0 Papirus:0

=== PIRAMIDA ===


In [ ]:
text = decoder.decode(raw_vector)
print(text)

=== STAN GLOBALNY ===
Aktywny Gracz: Gracz 1
Epoka: II
Pozycja Konfliktu: 0
Koniec Gry: False

--- AKTYWNY GRACZ ---
Monety: 0 | PZ: 0
Surowce: Glina:0 Kamień:0 Drewno:0 Szkło:0 Papirus:0

--- PRZECIWNIK ---
Monety: 0 | PZ: 0
Surowce: Glina:0 Kamień:0 Drewno:0 Szkło:0 Papirus:0

=== PIRAMIDA ===
